# BOCS-IDS — Final Submission Notebook
**Extends:** Chua & Salam, *Symmetry* 2023, 15, 1251  
**Training:** CIC-IDS2017 · **Testing:** CSE-CIC-IDS2018 (10%)

| Stage | Outputs |
|---|---|
| Stage 1 | Tables 2–7 · Figures 9–14 (base paper replication) |
| Stage 2 | Tables B1–B3 · Figures B1–B4 (BOCS-IDS seven-row ablation) |

**Folder layout:**
```
./cic/IDS2017/    ← CIC-IDS2017 CSV files
./cic/IDS2018/    ← CSE-CIC-IDS2018 CSV files
./results_cic/    ← Stage 1 outputs
./results_bocs/   ← Stage 2 outputs
```


In [1]:
import os, sys, time, importlib, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import scipy.stats as ss
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM

warnings.filterwarnings('ignore')
os.environ['CUDA_VISIBLE_DEVICES'] = '5,6,7'

sys.path.insert(0, '.')
import bocs_ids_experiment_v6 as exp
importlib.reload(exp)

TRAIN_DIR = './cic/IDS2017'
TEST_DIR  = './cic/IDS2018'
OUT_S1    = './results_cic'
OUT_S2    = './results_bocs'
OCSVM_CAP = 10_000

exp.TRAIN_DIR = TRAIN_DIR; exp.TEST_DIR = TEST_DIR
exp.OUT_S1    = OUT_S1;    exp.OUT_S2   = OUT_S2
os.makedirs(OUT_S1, exist_ok=True)
os.makedirs(OUT_S2, exist_ok=True)

print(f"Python {sys.version.split()[0]}  |  sklearn {__import__('sklearn').__version__}")
print(f"CUDA_VISIBLE_DEVICES = {os.environ['CUDA_VISIBLE_DEVICES']}")
print(f"TRAIN : {os.path.abspath(TRAIN_DIR)}")
print(f"TEST  : {os.path.abspath(TEST_DIR)}")


Python 3.12.11  |  sklearn 1.8.0
CUDA_VISIBLE_DEVICES = 5,6,7
TRAIN : /nfsshare/users/P127003042/RC/cic/IDS2017
TEST  : /nfsshare/users/P127003042/RC/cic/IDS2018


---
## Stage 1 — Base Paper Replication (Chua & Salam 2023 §5.2)

### Load, Align, Preprocess

In [2]:
df17_raw = exp.load_2017(TRAIN_DIR)
df18_raw = exp.load_2018(TEST_DIR)
df17_raw, df18_raw = exp.align_columns(df17_raw, df18_raw)
df17, b17c, b17t, a17c, a17t = exp.preprocess(df17_raw.copy(), 'CIC-IDS2017',    is_train=True)
df18, b18c, b18t, a18c, a18t = exp.preprocess(df18_raw.copy(), 'CSE-CIC-IDS2018', is_train=False)


    CIC-IDS2017 loaded: 2,830,743 rows, 79 cols
    Dropped 59 embedded header rows
    CSE-CIC-IDS2018 loaded (10%): 1,623,294 rows, 84 cols
    Shared columns: 78  |  Dropped (only in 2017): 1
    CIC-IDS2017: 649,762 rows after 1:1 balance (324,881 per class)
    CSE-CIC-IDS2018: 336,916 rows after 1:1 balance (168,458 per class)


### Table 2 — CIC-IDS2017 Class Distribution

In [3]:
exp.print_table_dist(b17c, b17t, a17c, a17t, 'CIC-IDS2017', tnum=2)



TABLE 2: Class distribution of CIC-IDS2017
                                                        Before Cleaning   After Cleaning & Resampling
  Classes                                     No. of Rows      (%)     No. of Rows      (%)
  --------------------------------------------------------------------------------------------------------
  BENIGN                                        2,273,097  80.3004%         324,881  50.0000%
  DoS Hulk                                        231,073   8.1630%         100,000  15.3903%
  PortScan                                        158,930   5.6144%         100,000  15.3903%
  DDoS                                            128,027   4.5227%         100,000  15.3903%
  DoS GoldenEye                                    10,293   0.3636%          10,293   1.5841%
  FTP-Patator                                       7,938   0.2804%           7,938   1.2217%
  SSH-Patator                                       5,897   0.2083%           5,897   0.907

### Table 3 — CSE-CIC-IDS2018 Class Distribution

In [4]:
exp.print_table_dist(b18c, b18t, a18c, a18t,
    'CSE-CIC-IDS2018 dataset (10% of entire dataset)', tnum=3)



TABLE 3: Class distribution of CSE-CIC-IDS2018 dataset (10% of entire dataset)
                                                        Before Cleaning   After Cleaning & Resampling
  Classes                                     No. of Rows      (%)     No. of Rows      (%)
  --------------------------------------------------------------------------------------------------------
  Benign                                        1,349,108  83.1093%         168,458  50.0000%
  DDOS attack-HOIC                                 68,130   4.1970%          68,130  20.2217%
  DDoS attacks-LOIC-HTTP                           57,587   3.5475%          57,587  17.0924%
  DoS attacks-Hulk                                 45,809   2.8220%          45,809  13.5966%
  Bot                                              28,581   1.7607%          28,581   8.4831%
  FTP-BruteForce                                   19,452   1.1983%          19,452   5.7735%
  SSH-Bruteforce                                   18,9

### Feature Matrices

In [5]:
feat_cols = [c for c in df17.columns if c != 'label']
df17_num  = df17[feat_cols].select_dtypes(include=[np.number])
feat_cols = list(df17_num.columns)

X_all = np.nan_to_num(df17_num.values.astype(float))
y_all = df17['label'].values
X_te  = np.nan_to_num(df18[feat_cols].reindex(columns=feat_cols).values.astype(float))
y_te  = df18['label'].values

scaler  = StandardScaler()
X_all_s = scaler.fit_transform(X_all)
X_te_s  = scaler.transform(X_te)

print(f"Training matrix : {X_all_s.shape}")
print(f"Test matrix     : {X_te_s.shape}")
print(f"Train  benign={int((y_all==0).sum()):,}  malicious={int((y_all==1).sum()):,}")
print(f"Test   benign={int((y_te==0).sum()):,}   malicious={int((y_te==1).sum()):,}")


Training matrix : (649762, 77)
Test matrix     : (336916, 77)
Train  benign=324,881  malicious=324,881
Test   benign=168,458   malicious=168,458


### Figure 9 — Feature Importance & Figure 10 — Accuracy vs Features

In [6]:
ranked_feats, ranked_imp = exp.feature_selection(X_all_s, y_all, feat_cols)
exp.fig9_importance(ranked_feats, ranked_imp, OUT_S1)

top_feats = ranked_feats[:exp.TOP_N]
top_idx   = [feat_cols.index(f) for f in top_feats]

exp.fig10_features_vs_accuracy(X_all_s, y_all, ranked_feats, feat_cols, OUT_S1)

X_tr   = X_all_s[:, top_idx]
X_test = X_te_s[:,  top_idx]

print(f"Top {exp.TOP_N} features selected:")
for i, (f, v) in enumerate(zip(top_feats, ranked_imp[:exp.TOP_N]), 1):
    print(f"  {i:2}. {f:<40} {v:.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 4))
for ax, f, t in [
    (ax1, f'{OUT_S1}/fig09_feature_importance.png',    'Figure 9: Feature Importance'),
    (ax2, f'{OUT_S1}/fig10_features_vs_accuracy.png',  'Figure 10: Accuracy vs Features'),
]:
    try:
        ax.imshow(mpimg.imread(f)); ax.axis('off'); ax.set_title(t, fontsize=11)
    except FileNotFoundError:
        ax.text(0.5, 0.5, 'not found', ha='center', transform=ax.transAxes); ax.axis('off')
plt.tight_layout()
plt.show()


    Saved ./results_cic/fig09_feature_importance.png
    Saved ./results_cic/fig10_features_vs_accuracy.png
Top 11 features selected:
   1. Packet Length Variance                   0.0601
   2. Max Packet Length                        0.0540
   3. Average Packet Size                      0.0526
   4. Destination Port                         0.0508
   5. Packet Length Std                        0.0487
   6. Bwd Packet Length Mean                   0.0424
   7. Fwd Packet Length Max                    0.0398
   8. Bwd Packet Length Std                    0.0386
   9. Bwd Packet Length Max                    0.0373
  10. Bwd Packet Length Min                    0.0359
  11. Avg Bwd Segment Size                     0.0354


### Table 4 — 5-Fold Cross-Validation

In [7]:
models = exp.get_models()
kf     = StratifiedKFold(n_splits=5, shuffle=True, random_state=exp.RS)
cv_results = {}

for name, mdl in models.items():
    print(f"  {name}...", end=' ', flush=True)
    Xcv, ycv = exp.cap_sample(X_tr, y_all, exp.get_cap(name))
    scores   = cross_val_score(mdl, Xcv, ycv, cv=kf,
                               scoring='accuracy', n_jobs=exp.N_JOBS)
    cv_results[name] = scores
    print(f"{scores.mean():.4f} ± {scores.std():.4f}")

exp.print_table4_cv(cv_results)


  Decision Tree... 0.9832 ± 0.0004
  Random Forest... 0.9849 ± 0.0004
  Support Vector Machine... 0.9383 ± 0.0013
  Naive Bayes... 0.7155 ± 0.0005
  Artificial Neural Network... 0.9244 ± 0.0028
  Deep Neural Network... 0.9566 ± 0.0064

TABLE 4: 5-fold cross-validation accuracy on CIC-IDS2017
  Model                            Fold    Accuracy   Mean Accuracy     Std Dev
  ------------------------------------------------------------------------------
  Decision Tree                  Fold-1      0.9824          0.9832      0.0004
                                 Fold-2      0.9834
                                 Fold-3      0.9829
                                 Fold-4      0.9836
                                 Fold-5      0.9835
  ------------------------------------------------------------------------------
  Random Forest                  Fold-1      0.9842          0.9849      0.0004
                                 Fold-2      0.9849
                                 Fold-3      

### Table 5 — Comparison vs Literature

In [8]:
exp.print_table5(cv_results)



TABLE 5: Accuracy comparison for the CIC dataset
  ML Models                       This Work   Kostas[42]  Vinayakumar[29]
  ----------------------------------------------------------------------
  Decision Tree                        0.98         0.95             0.94
  Random Forest                        0.98         0.94             0.94
  Support Vector Machine               0.94            -             0.80
  Naive Bayes                          0.72         0.87             0.31
  Artificial Neural Network            0.92         0.97             0.96
  Deep Neural Network                  0.96            -             0.94


### Tables 6 & 7 — Final Model Performance

In [9]:
perm  = np.random.RandomState(exp.RS).permutation(len(X_tr))
split = int(0.7 * len(X_tr))
X_70, y_70 = X_tr[perm[:split]], y_all[perm[:split]]
X_30, y_30 = X_tr[perm[split:]], y_all[perm[split:]]

r_train, r_test = {}, {}
train_times, pred_times = {}, {}

for name, mdl in models.items():
    print(f"  {name}...", end=' ', flush=True)
    Xfit, yfit = exp.cap_sample(X_70, y_70, exp.get_cap(name))
    t0 = time.time(); mdl.fit(Xfit, yfit); train_times[name] = time.time() - t0
    r_train[name] = exp.get_metrics(mdl, X_30, y_30)
    t0 = time.time(); r_test[name] = exp.get_metrics(mdl, X_test, y_te)
    pred_times[name] = time.time() - t0
    print(f"acc={r_test[name]['accuracy']:.4f}  rec_m={r_test[name]['malicious']['recall']:.4f}")

exp.print_table_perf(r_train, 'CIC-IDS2017 dataset',     tnum=6)
exp.print_table_perf(r_test,  'CSE-CIC-IDS2018 dataset', tnum=7)


  Decision Tree... acc=0.7346  rec_m=0.5195
  Random Forest... acc=0.7323  rec_m=0.5196
  Support Vector Machine... acc=0.8476  rec_m=0.7774
  Naive Bayes... acc=0.4913  rec_m=0.0018
  Artificial Neural Network... acc=0.7665  rec_m=0.6540
  Deep Neural Network... acc=0.7219  rec_m=0.4989

TABLE 6: Performance of models on CIC-IDS2017 dataset
  Models                           Accuracy  Precision     Recall   F1-Score  Class
  ------------------------------------------------------------------------------
  Decision Tree                      0.9833     0.9949     0.9715     0.9831  benign
                                                0.9723     0.9951     0.9836  malicious
  ------------------------------------------------------------------------------
  Random Forest                      0.9849     0.9975     0.9721     0.9846  benign
                                                0.9729     0.9976     0.9851  malicious
  --------------------------------------------------------------

### Figures 11 & 12 — Confusion Matrices

In [10]:
exp.fig_cm(r_train, 'CIC-IDS2017',     fignum=11, out_dir=OUT_S1)
exp.fig_cm(r_test,  'CSE-CIC-IDS2018', fignum=12, out_dir=OUT_S1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))
for ax, f, t in [
    (ax1, f'{OUT_S1}/fig11_cm_CIC_IDS2017.png',     'Figure 11: Confusion Matrices — CIC-IDS2017'),
    (ax2, f'{OUT_S1}/fig12_cm_CSE_CIC_IDS2018.png', 'Figure 12: Confusion Matrices — CSE-CIC-IDS2018'),
]:
    try:
        ax.imshow(mpimg.imread(f)); ax.axis('off'); ax.set_title(t, fontsize=11)
    except FileNotFoundError:
        ax.text(0.5, 0.5, 'not found', ha='center', transform=ax.transAxes); ax.axis('off')
plt.tight_layout()
plt.show()


    Saved ./results_cic/fig11_cm_CIC_IDS2017.png
    Saved ./results_cic/fig12_cm_CSE_CIC_IDS2018.png


### Figure 13 — Accuracy & F1 Comparison  ·  Figure 14 — Training & Prediction Time

In [11]:
exp.fig13_acc_f1(r_train, r_test, OUT_S1)
exp.fig14_time(train_times, pred_times, OUT_S1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 5))
for ax, f, t in [
    (ax1, f'{OUT_S1}/fig13_accuracy_f1.png', 'Figure 13: Accuracy & F1-Score'),
    (ax2, f'{OUT_S1}/fig14_time.png',         'Figure 14: Training & Prediction Time'),
]:
    try:
        ax.imshow(mpimg.imread(f)); ax.axis('off'); ax.set_title(t, fontsize=11)
    except FileNotFoundError:
        ax.text(0.5, 0.5, 'not found', ha='center', transform=ax.transAxes); ax.axis('off')
plt.tight_layout()
plt.show()
print("Stage 1 complete.")


    Saved ./results_cic/fig13_accuracy_f1.png
    Saved ./results_cic/fig14_time.png
Stage 1 complete.


### Comparison with Base Paper

In [12]:
paper = {
    'Decision Tree':             {'cv': 0.9970, 'acc': 0.5942, 'rec_m': 0.2208},
    'Random Forest':             {'cv': 0.9972, 'acc': 0.5949, 'rec_m': 0.2214},
    'Support Vector Machine':    {'cv': 0.9650, 'acc': 0.7559, 'rec_m': 0.6063},
    'Naive Bayes':               {'cv': 0.7311, 'acc': 0.4972, 'rec_m': 0.0003},
    'Artificial Neural Network': {'cv': 0.9693, 'acc': 0.7000, 'rec_m': 0.4959},
    'Deep Neural Network':       {'cv': 0.9772, 'acc': 0.6518, 'rec_m': 0.3689},
}

print(f"  {'Model':<30} {'CV Ours':>8} {'CV Paper':>9} {'Delta':>7} | "
      f"{'Acc Ours':>9} {'Acc Paper':>9} {'Delta':>7} | "
      f"{'RecM Ours':>9} {'RecM Paper':>10} {'Delta':>7}")
print(f"  {'-'*100}")
for name in exp.MODEL_NAMES:
    p  = paper[name]
    cv = cv_results[name].mean()
    ac = r_test[name]['accuracy']
    rm = r_test[name]['malicious']['recall']
    print(f"  {name:<30} {cv:>8.4f} {p['cv']:>9.4f} {cv-p['cv']:>+7.4f} | "
          f"{ac:>9.4f} {p['acc']:>9.4f} {ac-p['acc']:>+7.4f} | "
          f"{rm:>9.4f} {p['rec_m']:>10.4f} {rm-p['rec_m']:>+7.4f}")


  Model                           CV Ours  CV Paper   Delta |  Acc Ours Acc Paper   Delta | RecM Ours RecM Paper   Delta
  ----------------------------------------------------------------------------------------------------
  Decision Tree                    0.9832    0.9970 -0.0138 |    0.7346    0.5942 +0.1404 |    0.5195     0.2208 +0.2987
  Random Forest                    0.9849    0.9972 -0.0123 |    0.7323    0.5949 +0.1374 |    0.5196     0.2214 +0.2982
  Support Vector Machine           0.9383    0.9650 -0.0267 |    0.8476    0.7559 +0.0917 |    0.7774     0.6063 +0.1711
  Naive Bayes                      0.7155    0.7311 -0.0156 |    0.4913    0.4972 -0.0059 |    0.0018     0.0003 +0.0015
  Artificial Neural Network        0.9244    0.9693 -0.0449 |    0.7665    0.7000 +0.0665 |    0.6540     0.4959 +0.1581
  Deep Neural Network              0.9566    0.9772 -0.0206 |    0.7219    0.6518 +0.0701 |    0.4989     0.3689 +0.1300


---
## Stage 2 — BOCS-IDS Seven-Row Ablation

### Helpers

In [13]:
def cohen_d(s_b, s_m):
    pooled = np.sqrt((s_b.std()**2 + s_m.std()**2) / 2)
    return float(abs(s_b.mean() - s_m.mean()) / pooled) if pooled > 0 else 0.

benign_mask   = y_all == 0
X_benign_full = X_all_s[benign_mask]
print(f"Benign training flows: {X_benign_full.shape[0]:,} x {X_benign_full.shape[1]}")


Benign training flows: 324,881 x 77


### A0 — RF-11 Baseline

In [14]:
rf_11 = RandomForestClassifier(n_estimators=100, random_state=exp.RS, n_jobs=exp.N_JOBS)
rf_11.fit(X_tr, y_all)
r_rf11   = exp.get_metrics(rf_11, X_test, y_te)
base_rec = r_rf11['malicious']['recall']
print(f"A0  acc={r_rf11['accuracy']:.4f}  rec_m={base_rec:.4f}  f1_m={r_rf11['malicious']['f1']:.4f}")


A0  acc=0.7323  rec_m=0.5196  f1_m=0.6600


### A1 — BOCS-RF-11: Benign-Only IF, Top-11, Raw Score

In [15]:
t0 = time.time()
if_11 = IsolationForest(n_estimators=300, contamination='auto',
                         random_state=exp.RS, n_jobs=exp.N_JOBS)
if_11.fit(X_tr[benign_mask])
s_tr_11 = if_11.decision_function(X_tr).reshape(-1, 1)
s_te_11 = if_11.decision_function(X_test).reshape(-1, 1)
d11 = cohen_d(if_11.decision_function(X_test[y_te == 0]),
              if_11.decision_function(X_test[y_te == 1]))
print(f"IF-11  {time.time()-t0:.1f}s  Cohen d={d11:.4f}")

rf_b11 = RandomForestClassifier(n_estimators=100, random_state=exp.RS, n_jobs=exp.N_JOBS)
rf_b11.fit(np.hstack([X_tr, s_tr_11]), y_all)
r_bocs11 = exp.get_metrics(rf_b11, np.hstack([X_test, s_te_11]), y_te)
print(f"A1   acc={r_bocs11['accuracy']:.4f}  rec_m={r_bocs11['malicious']['recall']:.4f}  "
      f"Delta={r_bocs11['malicious']['recall']-base_rec:+.4f}")


IF-11  16.3s  Cohen d=0.0995
A1   acc=0.7322  rec_m=0.5193  Delta=-0.0003


### A1\* and A1\*B — Benign-Only IF, Full 77 Features

In [16]:
t0 = time.time()
if_full = IsolationForest(n_estimators=300, contamination='auto',
                           random_state=exp.RS, n_jobs=exp.N_JOBS)
if_full.fit(X_benign_full)
print(f"IF-full  {time.time()-t0:.1f}s  ({X_benign_full.shape[0]:,} x {X_benign_full.shape[1]})")

s_tr_if_raw = if_full.decision_function(X_all_s).reshape(-1, 1)
s_te_if_raw = if_full.decision_function(X_te_s).reshape(-1, 1)

s_tr_if, s_te_if, sign_if = exp.sign_correct_and_standardise(
    s_tr_if_raw, s_te_if_raw, y_all, 'A1*')
d_if = cohen_d(s_te_if[y_te == 0].flatten(), s_te_if[y_te == 1].flatten())
print(f"Cohen d (A1*) = {d_if:.4f}  sign={sign_if:+.0f}")

rf_bfull = RandomForestClassifier(n_estimators=100, random_state=exp.RS, n_jobs=exp.N_JOBS)
rf_bfull.fit(np.hstack([X_tr, s_tr_if]), y_all)
r_bocs_full = exp.get_metrics(rf_bfull, np.hstack([X_test, s_te_if]), y_te)
print(f"A1*  acc={r_bocs_full['accuracy']:.4f}  rec_m={r_bocs_full['malicious']['recall']:.4f}  "
      f"Delta={r_bocs_full['malicious']['recall']-base_rec:+.4f}")

bt     = if_full.decision_function(X_all_s[benign_mask])
thr_if = bt.mean() - 2 * bt.std()
flag_tr_if = (s_tr_if_raw < thr_if).astype(float)
flag_te_if = (s_te_if_raw < thr_if).astype(float)
print(f"IF threshold={thr_if:.4f}")
print(f"Train flag=1  benign={flag_tr_if[y_all==0].mean()*100:.1f}%  malicious={flag_tr_if[y_all==1].mean()*100:.1f}%")
print(f"Test  flag=1  benign={flag_te_if[y_te==0].mean()*100:.1f}%  malicious={flag_te_if[y_te==1].mean()*100:.1f}%")

rf_flag = RandomForestClassifier(n_estimators=100, random_state=exp.RS, n_jobs=exp.N_JOBS)
rf_flag.fit(np.hstack([X_tr, flag_tr_if]), y_all)
r_bocs_flag = exp.get_metrics(rf_flag, np.hstack([X_test, flag_te_if]), y_te)
print(f"A1*B acc={r_bocs_flag['accuracy']:.4f}  rec_m={r_bocs_flag['malicious']['recall']:.4f}  "
      f"Delta={r_bocs_flag['malicious']['recall']-base_rec:+.4f}")


IF-full  1.4s  (324,881 x 77)
    [A1*] Training benign mean=0.1264  malicious mean=0.0209
    [A1*] Direction correct — no flip needed
    [A1*] Standardisation: mean=0.0736  std=0.1160  sign=+1
Cohen d (A1*) = 0.2011  sign=+1
A1*  acc=0.5681  rec_m=0.1418  Delta=-0.3778
IF threshold=-0.0157
Train flag=1  benign=6.7%  malicious=47.3%
Test  flag=1  benign=7.1%  malicious=1.6%
A1*B acc=0.7325  rec_m=0.5199  Delta=+0.0002


### A2 — All-Data IF Control

In [17]:
if_all = IsolationForest(n_estimators=300, contamination=0.5,
                          random_state=exp.RS, n_jobs=exp.N_JOBS)
if_all.fit(X_all_s)
s_tr_a2r = if_all.decision_function(X_all_s).reshape(-1, 1)
s_te_a2r = if_all.decision_function(X_te_s).reshape(-1, 1)
s_tr_a2, s_te_a2, _ = exp.sign_correct_and_standardise(
    s_tr_a2r, s_te_a2r, y_all, 'A2')
rf_a2 = RandomForestClassifier(n_estimators=100, random_state=exp.RS, n_jobs=exp.N_JOBS)
rf_a2.fit(np.hstack([X_tr, s_tr_a2]), y_all)
r_a2 = exp.get_metrics(rf_a2, np.hstack([X_test, s_te_a2]), y_te)
print(f"A2   acc={r_a2['accuracy']:.4f}  rec_m={r_a2['malicious']['recall']:.4f}  "
      f"Delta={r_a2['malicious']['recall']-base_rec:+.4f}")


    [A2] Training benign mean=-0.0168  malicious mean=-0.0356
    [A2] Direction correct — no flip needed
    [A2] Standardisation: mean=-0.0262  std=0.0646  sign=+1
A2   acc=0.5662  rec_m=0.1489  Delta=-0.3707


### A3 & A3B — One-Class SVM (OCSVM_CAP=10,000)

In [18]:
rng       = np.random.RandomState(exp.RS)
idx_oc    = rng.choice(len(X_benign_full), OCSVM_CAP, replace=False)
X_oc_tr   = X_benign_full[idx_oc]
print(f"OCSVM training on {X_oc_tr.shape[0]:,} / {X_benign_full.shape[0]:,} benign flows")

t0 = time.time()
ocsvm = OneClassSVM(kernel='rbf', nu=0.05, gamma='scale')
ocsvm.fit(X_oc_tr)
print(f"OCSVM  {time.time()-t0:.1f}s")

s_tr_oc_raw = ocsvm.decision_function(X_all_s).reshape(-1, 1)
s_te_oc_raw = ocsvm.decision_function(X_te_s).reshape(-1, 1)
s_tr_oc, s_te_oc, sign_oc = exp.sign_correct_and_standardise(
    s_tr_oc_raw, s_te_oc_raw, y_all, 'A3')
d_ocsvm = cohen_d(s_te_oc[y_te == 0].flatten(), s_te_oc[y_te == 1].flatten())
print(f"Cohen d (A3) = {d_ocsvm:.4f}  sign={sign_oc:+.0f}")
print(f"Progression  A1={d11:.4f}  A1*={d_if:.4f}  A3={d_ocsvm:.4f}")

t0 = time.time()
rf_a3 = RandomForestClassifier(n_estimators=100, random_state=exp.RS, n_jobs=exp.N_JOBS)
rf_a3.fit(np.hstack([X_tr, s_tr_oc]), y_all)
r_a3 = exp.get_metrics(rf_a3, np.hstack([X_test, s_te_oc]), y_te)
print(f"A3   acc={r_a3['accuracy']:.4f}  rec_m={r_a3['malicious']['recall']:.4f}  "
      f"Delta={r_a3['malicious']['recall']-base_rec:+.4f}  clf={time.time()-t0:.1f}s")

oc_bt  = ocsvm.decision_function(X_oc_tr)
thr_oc = oc_bt.mean() - 2 * oc_bt.std()
flag_tr_oc = (s_tr_oc_raw < thr_oc).astype(float)
flag_te_oc = (s_te_oc_raw < thr_oc).astype(float)
print(f"OCSVM threshold={thr_oc:.4f}")
print(f"Train flag=1  benign={flag_tr_oc[y_all==0].mean()*100:.1f}%  malicious={flag_tr_oc[y_all==1].mean()*100:.1f}%")
print(f"Test  flag=1  benign={flag_te_oc[y_te==0].mean()*100:.1f}%  malicious={flag_te_oc[y_te==1].mean()*100:.1f}%")

rf_a3b = RandomForestClassifier(n_estimators=100, random_state=exp.RS, n_jobs=exp.N_JOBS)
rf_a3b.fit(np.hstack([X_tr, flag_tr_oc]), y_all)
r_a3b = exp.get_metrics(rf_a3b, np.hstack([X_test, flag_te_oc]), y_te)
print(f"A3B  acc={r_a3b['accuracy']:.4f}  rec_m={r_a3b['malicious']['recall']:.4f}  "
      f"Delta={r_a3b['malicious']['recall']-base_rec:+.4f}")


OCSVM training on 10,000 / 324,881 benign flows
OCSVM  0.8s
    [A3] Training benign mean=5.8233  malicious mean=-0.5040
    [A3] Direction correct — no flip needed
    [A3] Standardisation: mean=2.6597  std=6.9092  sign=+1
Cohen d (A3) = 0.3928  sign=+1
Progression  A1=0.0995  A1*=0.2011  A3=0.3928
A3   acc=0.6605  rec_m=0.3363  Delta=-0.1833  clf=40.0s
OCSVM threshold=-1.2878
Train flag=1  benign=3.6%  malicious=46.2%
Test  flag=1  benign=27.8%  malicious=49.2%
A3B  acc=0.7324  rec_m=0.5192  Delta=-0.0005


### Table B1 — Full Ablation Comparison

In [19]:
exp.print_table_b1(r_rf11, r_bocs11, r_bocs_full, r_bocs_flag,
                   r_a2, r_a3, r_a3b)



TABLE B1: BOCS ablation comparison on CSE-CIC-IDS2018
  Model                                              Accuracy   Prec(B)   Rec(B)   F1(B)   Prec(M)   Rec(M)   F1(M)     AUC
  ----------------------------------------------------------------------------------------------------------------------
  A0:   RF-11 (no IF, baseline)                        0.7323    0.6630   0.9450  0.7792    0.9042   0.5196  0.6600  0.7323
  A1:   BOCS-RF-11 (top-11 IF, raw)                    0.7322    0.6628   0.9451  0.7792    0.9043   0.5193  0.6597  0.7322
  A1*:  BOCS-RF-full (IF, sign-corr+std)               0.5681    0.5368   0.9945  0.6972    0.9624   0.1418  0.2472  0.5681
  A1*B: BOCS-RF-full (IF, binary flag) ◄               0.7325    0.6631   0.9451  0.7794    0.9045   0.5199  0.6603  0.7325
  A2:   RF + all-data IF, sign-corr+std                0.5662    0.5361   0.9835  0.6939    0.9002   0.1489  0.2555  0.5662
  A3:   BOCS-RF-full (OCSVM, sign-corr+std) ◄          0.6605    0.5974   0.9847

### Table B2 — Ablation with Cohen's d and Delta rec_m

In [20]:
exp.print_table_b2(r_rf11, r_bocs11, r_bocs_full, r_bocs_flag,
                   r_a2, r_a3, r_a3b, d11, d_if, d_ocsvm)



TABLE B2: Ablation Study on CSE-CIC-IDS2018
  Configuration                                          Accuracy   Rec(Mal)   F1(Mal)     AUC   Cohen d    Δrec_m
  --------------------------------------------------------------------------------------------------------------
  A0:   RF-11 only (baseline)                              0.7323     0.5196    0.6600  0.7323         -   +0.0000
  A1:   BOCS-RF-11 (top-11 IF, raw)                        0.7322     0.5193    0.6597  0.7322    0.0995   -0.0003
  A1*:  BOCS-RF-full (IF, sign-corr+std)                   0.5681     0.1418    0.2472  0.5681    0.2011   -0.3778
  A1*B: BOCS-RF-full (IF, binary flag)  ◄                  0.7325     0.5199    0.6603  0.7325    0.2011   +0.0002
  A2:   RF + all-data IF, sign-corr+std                    0.5662     0.1489    0.2555  0.5662         -   -0.3707
  A3:   BOCS-RF-full (OCSVM, sign-corr+std) ◄              0.6605     0.3363    0.4976  0.6605    0.3928   -0.1833
  A3B:  BOCS-RF-full (OCSVM, binary f

### Table B3 — McNemar's Test

In [21]:
candidates = {'A1*B': r_bocs_flag, 'A3': r_a3, 'A3B': r_a3b}
best_name  = max(candidates, key=lambda k: candidates[k]['malicious']['recall'])
best_r     = candidates[best_name]
print(f"Best BOCS config: {best_name}  rec_m={best_r['malicious']['recall']:.4f}")

yp_best = best_r['pred']
mcn = {}
for name in exp.MODEL_NAMES:
    yp_base = r_test[name]['pred']
    c1 = yp_best == y_te
    c2 = yp_base == y_te
    b  = int(np.sum( c1 & ~c2))
    c_ = int(np.sum(~c1 &  c2))
    if b + c_ == 0:
        mcn[name] = {'b': 0, 'c': 0, 'chi2': 0., 'p': 1., 'sig': 'ns'}
    else:
        chi2 = float((abs(b - c_) - 1)**2 / (b + c_))
        p    = float(ss.chi2.sf(chi2, df=1))
        sig  = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        mcn[name] = {'b': b, 'c': c_, 'chi2': chi2, 'p': p, 'sig': sig}

exp.print_table_b3(mcn)


Best BOCS config: A1*B  rec_m=0.5199

TABLE B3: McNemar's Test — BOCS-RF-flag (A1*B) vs Stage-1 Baselines
  (A1*B = binary novelty flag — direction-robust, rec_m=+0.0002 vs baseline)
  Comparison                                  b        c      chi2      p-value    sig
  ------------------------------------------------------------------------------------
  BOCS-flag vs Decision Tree              3,697    4,403    61.361     0.000000    ***
  BOCS-flag vs Random Forest                 84       13    50.515     0.000000    ***
  BOCS-flag vs Support Vector Machine     9,064   47,828 26410.922     0.000000    ***
  BOCS-flag vs Naive Bayes               90,793    9,526 65831.625     0.000000    ***
  BOCS-flag vs Artificial Neural Network   19,944   31,388  2550.889     0.000000    ***
  BOCS-flag vs Deep Neural Network        8,392    4,831   958.451     0.000000    ***
  b = BOCS-flag correct & baseline wrong  |  c = BOCS-flag wrong & baseline correct
  *** p<0.001  ** p<0.01  * p<0.05 

### Figure B1 — IF vs OCSVM Score Distributions

In [22]:
exp.figb1_if_scores(if_full, ocsvm, X_te_s, y_te, OUT_S2)
fig, ax = plt.subplots(figsize=(14, 5))
try:
    ax.imshow(mpimg.imread(f'{OUT_S2}/figB1_score_distributions.png'))
    ax.axis('off')
    ax.set_title('Figure B1: IF vs OCSVM Score Distributions', fontsize=11)
except FileNotFoundError:
    ax.text(0.5, 0.5, 'not found', ha='center', transform=ax.transAxes); ax.axis('off')
plt.tight_layout()
plt.show()


    Saved ./results_bocs/figB1_score_distributions.png


### Figure B2 — Stage-1 Baselines vs Best BOCS Config

In [23]:
exp.figb2_comparison(r_test, best_r, OUT_S2)
fig, ax = plt.subplots(figsize=(14, 5))
try:
    ax.imshow(mpimg.imread(f'{OUT_S2}/figB2_stage1_vs_bocs.png'))
    ax.axis('off')
    ax.set_title(f'Figure B2: Stage-1 Baselines vs {best_name}', fontsize=11)
except FileNotFoundError:
    ax.text(0.5, 0.5, 'not found', ha='center', transform=ax.transAxes); ax.axis('off')
plt.tight_layout()
plt.show()


    Saved ./results_bocs/figB2_stage1_vs_bocs.png


### Figure B3 — Confusion Matrix of Best BOCS Config

In [24]:
exp.figb3_cm_bocs(best_r, OUT_S2)
fig, ax = plt.subplots(figsize=(5, 4))
try:
    ax.imshow(mpimg.imread(f'{OUT_S2}/figB3_bocs_confusion_matrix.png'))
    ax.axis('off')
    ax.set_title(f'Figure B3: {best_name} Confusion Matrix', fontsize=11)
except FileNotFoundError:
    ax.text(0.5, 0.5, 'not found', ha='center', transform=ax.transAxes); ax.axis('off')
plt.tight_layout()
plt.show()


    Saved ./results_bocs/figB3_bocs_confusion_matrix.png


### Figure B4 — Full Ablation Bar Chart

In [25]:
exp.figb4_ablation(r_rf11, r_bocs11, r_bocs_full, r_bocs_flag,
                   r_a2, r_a3, r_a3b, OUT_S2)
fig, ax = plt.subplots(figsize=(14, 5))
try:
    ax.imshow(mpimg.imread(f'{OUT_S2}/figB4_ablation.png'))
    ax.axis('off')
    ax.set_title('Figure B4: Full Ablation A0–A3B', fontsize=11)
except FileNotFoundError:
    ax.text(0.5, 0.5, 'not found', ha='center', transform=ax.transAxes); ax.axis('off')
plt.tight_layout()
plt.show()


    Saved ./results_bocs/figB4_ablation.png


---
### Final Summary

In [26]:
import glob

print("STAGE 1 OUTPUTS")
for p in sorted(glob.glob(f'{OUT_S1}/*.png')):
    print(f"  {os.path.basename(p)}")

print("\nSTAGE 2 OUTPUTS")
for p in sorted(glob.glob(f'{OUT_S2}/*.png')):
    print(f"  {os.path.basename(p)}")

print(f"\nCOHEN'S D PROGRESSION")
print(f"  A1  IF top-11 raw  : {d11:.4f}")
print(f"  A1* IF full        : {d_if:.4f}")
print(f"  A3  OCSVM full     : {d_ocsvm:.4f}")

print(f"\nABLATION RESULTS")
for tag, r in [
    ('A0  RF-11',    r_rf11),
    ('A1  BOCS-11',  r_bocs11),
    ('A1* IF-std',   r_bocs_full),
    ('A1*B IF-flag', r_bocs_flag),
    ('A2  allIF',    r_a2),
    ('A3  OC-std',   r_a3),
    ('A3B OC-flag',  r_a3b),
]:
    d = r['malicious']['recall'] - base_rec
    marker = ' <- BEST' if r is best_r else ''
    print(f"  {tag:<12} acc={r['accuracy']:.4f}  rec_m={r['malicious']['recall']:.4f}  "
          f"f1_m={r['malicious']['f1']:.4f}  Delta={d:+.4f}{marker}")


STAGE 1 OUTPUTS
  fig09_feature_importance.png
  fig10_features_vs_accuracy.png
  fig11_cm_CIC_IDS2017.png
  fig12_cm_CSE_CIC_IDS2018.png
  fig13_accuracy_f1.png
  fig14_time.png

STAGE 2 OUTPUTS
  figB1_if_score_distribution.png
  figB1_score_distributions.png
  figB2_stage1_vs_bocs.png
  figB3_bocs_confusion_matrix.png
  figB4_ablation.png

COHEN'S D PROGRESSION
  A1  IF top-11 raw  : 0.0995
  A1* IF full        : 0.2011
  A3  OCSVM full     : 0.3928

ABLATION RESULTS
  A0  RF-11    acc=0.7323  rec_m=0.5196  f1_m=0.6600  Delta=+0.0000
  A1  BOCS-11  acc=0.7322  rec_m=0.5193  f1_m=0.6597  Delta=-0.0003
  A1* IF-std   acc=0.5681  rec_m=0.1418  f1_m=0.2472  Delta=-0.3778
  A1*B IF-flag acc=0.7325  rec_m=0.5199  f1_m=0.6603  Delta=+0.0002 <- BEST
  A2  allIF    acc=0.5662  rec_m=0.1489  f1_m=0.2555  Delta=-0.3707
  A3  OC-std   acc=0.6605  rec_m=0.3363  f1_m=0.4976  Delta=-0.1833
  A3B OC-flag  acc=0.7324  rec_m=0.5192  f1_m=0.6598  Delta=-0.0005
